In [1]:
import osimport torchimport matplotlib.pyplot as pltimport jsonimport numpy as np

In [2]:
# data = torch.load('./test/stage_A_run_001_2.pt')# data = torch.load('./test/stage_A_run_002.pt')# data = torch.load('./test/stage_A_run_003_adjust_P.pt')data = torch.load('./test/stage_A_run_003_1_tilde_P.pt')# data = torch.load('./test/stage_A_run_004_increase_Rtotal.pt')# 从JSON文件读取基因注释# with open('./test/stage_A_world_run_001_2.json', 'r') as f:# with open('./test/stage_A_world_run_002.json', 'r') as f:# with open('./test/stage_A_world_run_003_adjust_P.json', 'r') as f:with open('./test/stage_A_world_run_003_1_tilde_P.json', 'r') as f:# with open('./test/stage_A_world_run_004_increase_Rtotal.json', 'r') as f:    world_data = json.load(f)# 提取基因类型映射gene_annotation = world_data['gene_annotation']gene_categories = gene_annotation['category']gene_to_macro = {int(k): v for k, v in gene_annotation['to_macro'].items()}gene_to_micro = {int(k): v for k, v in gene_annotation['to_micro'].items()}X_traj = data['X_traj']P_traj = data['P_traj']Z_traj = data['Z_traj']N_traj = data['N_traj']

In [5]:
T = X_traj.shape[0] - 1G = X_traj.shape[1]print(f'T = {T}, G = {G}')print(f'X_traj shape: {X_traj.shape}')print(f'P_traj shape: {P_traj.shape}')print(f'Z_traj shape: {Z_traj.shape}')print(f'N_traj shape: {N_traj.shape}')

T = 200, G = 50
X_traj shape: torch.Size([201, 50])
P_traj shape: torch.Size([201, 50])
Z_traj shape: torch.Size([201, 50])
N_traj shape: torch.Size([201])


In [ ]:
%matplotlib inlineimport matplotlib.pyplot as pltimport numpy as np# 预先将 Tensor 转为 numpy 数组X_data = X_traj.numpy()P_data = P_traj.numpy()Z_data = Z_traj.numpy()N_data = N_traj.numpy()T_steps, G_count = X_data.shapet_axis = np.arange(T_steps)fig, axes = plt.subplots(2, 2, figsize=(16, 10))ax1, ax2 = axes[0, 0], axes[0, 1]ax3, ax4 = axes[1, 0], axes[1, 1]# ==========================================# 动态且高对比度的颜色分配 (基于 Set1)# ==========================================cmap_set1 = plt.cm.Set1# Set1 颜色索引：0红, 1蓝, 2绿, 3紫, 4橙, 5黄, 6棕, 7粉, 8灰# 我们手工划分色系池，保证大类隔离且内部高对比度macro_color_pools = {    "Core Regulatory": [0, 4, 3, 6, 7], # 红, 橙, 紫, 棕, 粉 (高亮核心色)    "Cell Fate": [1, 2],                # 蓝, 绿 (外围冷色)    "Background": [8]                   # 灰 (背景色)}category_to_color = {}pool_cursors = {k: 0 for k in macro_color_pools.keys()}# ==========================================# 统一遍历所有基因，并在前三个子图中画线# ==========================================added_to_legend = set()for i in range(G_count):    macro_type = gene_to_macro[i]    micro_type = gene_to_micro[i]    category_label = f"{macro_type} - {micro_type}"        # 从对应的色系池中按顺序抓取高对比度颜色    if category_label not in category_to_color:        pool = macro_color_pools.get(macro_type, [8])        cursor = pool_cursors.get(macro_type, 0)        category_to_color[category_label] = cmap_set1(pool[cursor % len(pool)])        pool_cursors[macro_type] += 1            current_color = category_to_color[category_label]        # 背景基因视觉弱化    if macro_type == "Background":        alpha, linewidth, zorder = 0.65, 1.2, 1    else:        alpha, linewidth, zorder = 0.85, 1.8, 2            # 注册图例    label = category_label if category_label not in added_to_legend else None    if label:        added_to_legend.add(category_label)            ax1.plot(t_axis, X_data[:, i], color=current_color, alpha=alpha, linewidth=linewidth, zorder=zorder, label=label)    ax2.plot(t_axis, P_data[:, i], color=current_color, alpha=alpha, linewidth=linewidth, zorder=zorder)    ax3.plot(t_axis, Z_data[:, i], color=current_color, alpha=alpha, linewidth=linewidth, zorder=zorder)# ==========================================# 设置子图标题和坐标轴# ==========================================ax1.set_xlabel('Time (T)')ax1.set_ylabel('mRNA Expression')ax1.set_title('X_traj (mRNA) over Time')ax1.grid(True, linestyle='--', alpha=0.6)ax2.set_xlabel('Time (T)')ax2.set_ylabel('Protein Level')ax2.set_title('P_traj (Protein) over Time')ax2.grid(True, linestyle='--', alpha=0.6)ax3.set_xlabel('Time (T)')ax3.set_ylabel('Chromatin State')ax3.set_title('Z_traj (Chromatin) over Time')ax3.grid(True, linestyle='--', alpha=0.6)# ==========================================# 第四个子图：N_traj (单细胞数量/群体状态)# ==========================================ax4.plot(t_axis, N_data, color=cmap_set1(2), linewidth=2.5) # 使用绿色ax4.set_xlabel('Time (T)')ax4.set_ylabel('Cell Count / State')ax4.set_title('N_traj (Population State) over Time')ax4.grid(True, linestyle='--', alpha=0.6)# ==========================================# 全局排版与强制图例排序# ==========================================handles, labels = ax1.get_legend_handles_labels()# 强制按照生物学逻辑层级排序macro_priority = {    "Core Regulatory": 1,    "Cell Fate": 2,    "Background": 3}def legend_sort_key(item):    label_text = item[0]    mac = label_text.split(" - ")[0]    # 第一关键字为定义好的大类优先级，第二关键字为细分类的字母顺序    return (macro_priority.get(mac, 99), label_text)# # 按照自定义规则排序# sorted_pairs = sorted(zip(labels, handles), key=legend_sort_key)# sorted_labels, sorted_handles = zip(*sorted_pairs)fig.legend(handles, labels, loc='center left', bbox_to_anchor=(0.80, 0.5),            title="Gene Categories", fontsize=11, title_fontsize=13)plt.tight_layout(rect=[0, 0, 0.80, 1])plt.show()